## Executive Summary

This notebook implements the **star-of-slots (SOS) method** — the canonical winding layout engine for AC machines.

### What it does
Given a slot/pole/phase combination (Q, P, m), the SOS method:
1. Projects each slot's EMF phasor onto a circle divided into 2m sectors.
2. Assigns each slot conductor to a phase and current direction based on which sector its phasor falls in.
3. Builds the coil matrix used by winding factor and MMF analysis.

### When to use this module
- **Winding layout**: understand which slots carry which phase in which direction.
- **Feasibility check**: verify that a given (Q, P, m) combination supports a valid balanced winding.
- **Input to winding factors and MMF**: `build_coil_matrix` is the single output consumed by `winding_factors.py` and `mmf.py`.

> **Swappability:** this module is deliberately isolated as the layout engine. Future layout techniques (e.g. optimal assignment, genetic algorithm) can replace it without changing `winding_factors.py` or `mmf.py`.

## How It Fits Into Motor Design

```
01_winding_sos.ipynb   ←── YOU ARE HERE
(winding layout)
      │
      ▼
01_winding_factors.ipynb   (kp, kd, kw, harmonic spectrum)
      │
      ▼
01_winding_mmf.ipynb       (MMF waveform, harmonic amplitudes)
      │
      ▼
02_pmsm.ipynb              (torque, back-EMF)
```

**Depends on:** nothing (this is the foundation layer).

**Feeds into:** `01_winding_factors.ipynb`, `01_winding_mmf.ipynb`.

## What You Provide (Inputs)

| Parameter | Symbol | Description | Typical Range |
|-----------|--------|-------------|---------------|
| Stator slots | Q | Total slot count | 6 – 72 |
| Poles | P | Number of poles (even) | 2 – 20 |
| Phases | m | Phase count | 3 |
| Layers | layers | 1 = single-layer, 2 = double-layer | 1 or 2 |
| Coil span | w | Coil span in slots | 1 – Q//p |

## What You Get (Outputs)

| Result | Meaning |
|--------|---------|
| `build_star_of_slots` | Phasor angles for each slot |
| `assign_phases` | Phase/direction assignment per slot |
| `build_coil_matrix` | Integer occupancy matrix (layers × Q) |
| `winding_factor_sos` | |kw(ν)| via phasor sum |
| `get_basic_params` | Derived parameters (p, q, t, …) |
| `check_symmetry` | Validate symmetry/balance |
| `get_valid_coil_spans` | All mechanically valid coil spans |
| `is_valid_combination` | Boolean feasibility check |

## Abbreviations and Notation

| Symbol | Definition | Units |
|--------|-----------|-------|
| Q | Number of stator slots | — |
| P | Number of poles | — |
| p | Pole pairs, p = P/2 | — |
| m | Number of phases | — |
| t | Winding periodicity, t = gcd(Q, p) | — |
| q | Slots per pole per phase, q = Q/(2mp) | — |
| τp | Pole pitch in slots, τp = Q/P | slots |
| w | Coil span in slots | slots |
| α_e | Electrical slot pitch angle, α_e = p·(2π/Q) | rad |
| ν | Mechanical spatial harmonic order | — |

## Imports and Setup

In [ ]:
#| default_exp winding.sos

In [ ]:
#| export
from __future__ import annotations

import math
from functools import lru_cache
from fractions import Fraction
from math import gcd

import numpy as np

---

## Private Helpers

These internal functions support the SOS algorithm and are not part of the public API.

In [ ]:
#| export
def _lcm(a: int, b: int) -> int:
    """Least common multiple of two positive integers."""
    return a * b // gcd(a, b)


def _angular_distance(a: float, b: float) -> float:
    """
    Smallest angular distance between two angles (radians), in [0, π].
    """
    d = abs(a - b) % (2 * math.pi)
    return min(d, 2 * math.pi - d)

---

## `get_basic_params`: Winding Parameter Summary

### Theory

All downstream calculations start from a handful of derived parameters.
The winding periodicity *t* = gcd(Q, p) is the key symmetry parameter:
a valid winding repeats itself *t* times around the stator.

The slots-per-pole-per-phase *q* = Q/(2mp) classifies the winding:
- q ≥ 1 (integer): **distributed integer-slot winding** — low harmonic content.
- q < 1 (fractional): **fractional-slot concentrated winding (FSCW)** — compact end-turns, higher harmonic content.

**References:** Pyrhönen et al. (2008), §2.1.

In [ ]:
#| export
def get_basic_params(Q: int, P: int, m: int = 3) -> dict:
    """
    Compute basic winding parameters from slot/pole/phase count.

    Parameters
    ----------
    Q : int   Number of stator slots
    P : int   Number of poles (even)
    m : int   Number of phases (default 3)

    Returns
    -------
    dict with keys:
        p       : int   Pole pairs (P // 2)
        t       : int   Winding periodicity = gcd(Q, p)
        q       : Fraction   Slots per pole per phase = Q / (2mp)
        lcm_QP  : int   lcm(Q, P)
        alpha_e : float Electrical slot pitch (radians) = p · 2π/Q
        Q_phase : int   Slots per phase = Q // m  (integer part)

    Examples
    --------
    >>> p = get_basic_params(12, 10)
    >>> p['p'], p['t']
    (5, 1)
    >>> p = get_basic_params(24, 4)
    >>> p['q']
    Fraction(2, 1)
    """
    p = P // 2
    t = gcd(Q, p)
    q = Fraction(Q, 2 * m * p)
    lcm_QP = _lcm(Q, P)
    alpha_e = p * 2 * math.pi / Q
    Q_phase = Q // m
    return dict(p=p, t=t, q=q, lcm_QP=lcm_QP, alpha_e=alpha_e, Q_phase=Q_phase)

In [ ]:
from emachines.winding.sos import get_basic_params

params = get_basic_params(12, 10)
print("12s/10p (FSCW):")
for k, v in params.items():
    print(f"  {k:10s} = {v}")

print()
params = get_basic_params(24, 4)
print("24s/4p (integer-slot):")
for k, v in params.items():
    print(f"  {k:10s} = {v}")

---

## `build_star_of_slots`: EMF Phasor Angles

### Theory

The **star of slots** (also called the voltage star or EMF star) maps each stator slot to a phasor angle in the electrical domain.

For slot *i*, the electrical phasor angle is:

$$
\alpha_i = i \cdot \alpha_e \mod 2\pi, \qquad \alpha_e = p \cdot \frac{2\pi}{Q}
$$

where *p* = P/2 is the number of pole pairs and *α_e* is the electrical slot pitch.

Plotting all Q phasors reveals the "star" pattern that names the method. Slots whose phasors point in similar directions naturally carry the same phase current.

**Reference:** Bianchi & Bolognani (2002), IEEE Trans. Ind. Appl., 38(5).

In [ ]:
#| export
def build_star_of_slots(Q: int, P: int) -> np.ndarray:
    """
    Compute EMF phasor angles for all Q slots (electrical radians, mod 2π).

    Parameters
    ----------
    Q : int   Number of stator slots
    P : int   Number of poles

    Returns
    -------
    np.ndarray, shape (Q,), dtype float64
        Phasor angles α_i = i · p · 2π/Q mod 2π

    Examples
    --------
    >>> import numpy as np
    >>> angles = build_star_of_slots(6, 2)
    >>> np.round(angles, 4)
    array([0.    , 1.0472, 2.0944, 3.1416, 4.1888, 5.236 ])
    """
    p = P // 2
    alpha_e = p * 2 * math.pi / Q
    return np.array([(i * alpha_e) % (2 * math.pi) for i in range(Q)],
                    dtype=np.float64)

In [ ]:
from emachines.winding.sos import build_star_of_slots
import numpy as np

angles = build_star_of_slots(12, 10)
print("12s/10p phasor angles (degrees):")
for i, a in enumerate(angles):
    print(f"  Slot {i:2d}: {np.degrees(a):6.1f}°")

---

## `assign_phases`: Phase Assignment via Sector Map

### Theory

The 2π electrical circle is divided into **2m equal sectors** (width = π/m each).
Sectors alternate between forward (+) and return (−) conductors, and cycle through all m phases.

Each slot's phasor (from `build_star_of_slots`) is projected onto this sector map:
- The slot is assigned to the phase whose sector contains the phasor.
- Positive sector → forward conductor (+phase index).
- Negative sector → return conductor (−phase index).

For FSCW, multiple slots in the same phase produce phasors close together, forming tight "clusters" in the star diagram.

**Reference:** Pyrhönen et al. (2008), §2.3; Bianchi & Bolognani (2002).

In [ ]:
#| export
def _assign_slot_to_phase(angle: float, m: int) -> int:
    """
    Return the signed phase code for a slot at electrical angle `angle`.

    Uses floor-division sector assignment: the electrical circle [0, 2π) is
    divided into 2m sectors of width π/m each. Sector k contains angles
    [k·π/m, (k+1)·π/m). Phase codes: k < m → +(k+1), k >= m → -(k-m+1).

    Returns
    -------
    int in {+1…+m, -1…-m}
    """
    sector_width = math.pi / m
    # Small epsilon pushes exact-boundary angles into the right sector
    k = int((angle % (2 * math.pi) + 1e-9) / sector_width) % (2 * m)
    if k < m:
        return k + 1
    else:
        return -(k - m + 1)


In [ ]:
#| export
def assign_phases(Q: int, P: int, m: int = 3) -> np.ndarray:
    """
    Assign each slot to a phase and current direction using the star-of-slots.

    Parameters
    ----------
    Q : int   Number of stator slots
    P : int   Number of poles
    m : int   Number of phases (default 3)

    Returns
    -------
    np.ndarray, shape (Q,), dtype int
        Signed phase codes: +k = phase k forward, −k = phase k return.
        Phase index k ∈ {1, …, m}.

    Examples
    --------
    >>> assign_phases(6, 2, m=3)
    array([ 1,  2,  3, -1, -2, -3])
    """
    angles = build_star_of_slots(Q, P)
    return np.array([_assign_slot_to_phase(a, m) for a in angles], dtype=int)

In [ ]:
from emachines.winding.sos import assign_phases

codes = assign_phases(12, 10)
print("12s/10p phase assignment:")
names = {1: 'A+', 2: 'B+', 3: 'C+', -1: 'A-', -2: 'B-', -3: 'C-'}
for i, c in enumerate(codes):
    print(f"  Slot {i:2d}: {names[c]}")

---

## `build_coil_matrix`: Conductor Occupancy Matrix

### Theory

A coil connects a **go** conductor in one slot to a **return** conductor `w` slots away (coil span *w*).
For a single-layer winding, only go conductors are placed; the return side falls in a dedicated slot.
For a double-layer winding, every slot has two conductor layers — a go side from one coil and a return from another.

The coil matrix shape is `(layers, Q)`:
- Value `+k` : phase k, forward current direction.
- Value `−k` : phase k, return current direction.
- Value `0`  : empty slot (only possible in single-layer windings).

This matrix is the single interface consumed by `winding_factors.py` and `mmf.py`.

**References:** Hanselman (2003), Ch. 4; Pyrhönen et al. (2008), §2.3.

In [ ]:
#| export
def build_coil_matrix(
    Q: int,
    P: int,
    m: int = 3,
    layers: int = 1,
    w: int | None = None,
) -> np.ndarray:
    """
    Build the (layers × Q) conductor occupancy matrix.

    Parameters
    ----------
    Q      : int         Number of stator slots
    P      : int         Number of poles
    m      : int         Number of phases (default 3)
    layers : int         1 (single-layer) or 2 (double-layer)
    w      : int | None  Coil span in slots. None → Q // (P // 2).

    Returns
    -------
    np.ndarray, shape (layers, Q), dtype int
        +k = phase k forward, −k = phase k return, 0 = empty.

    Examples
    --------
    >>> build_coil_matrix(6, 2, m=3, layers=1).tolist()
    [[1, 2, 3, -1, -2, -3]]
    >>> build_coil_matrix(6, 2, m=3, layers=2, w=3).shape
    (2, 6)
    """
    p = P // 2
    if w is None:
        w = max(1, Q // P)  # pole pitch in slots (τp = Q/P, floored)

    phase_codes = assign_phases(Q, P, m)   # go-conductor assignment

    matrix = np.zeros((layers, Q), dtype=int)

    if layers == 1:
        # Place go conductors in all Q slots (single-layer → no return needed here;
        # the return side is implicit in the slot diametrically opposite).
        for i in range(Q):
            matrix[0, i] = phase_codes[i]
    else:
        # Double-layer: top layer = go conductors, bottom layer = return conductors
        # shifted by coil span w.
        for i in range(Q):
            go_code  = phase_codes[i]
            ret_slot = (i + w) % Q
            matrix[0, i]        = +go_code
            matrix[1, ret_slot] = -go_code

    return matrix

In [ ]:
from emachines.winding.sos import build_coil_matrix
import numpy as np

print("6s/2p single-layer:")
m1 = build_coil_matrix(6, 2, m=3, layers=1)
print(m1)

print()
print("12s/10p double-layer:")
m2 = build_coil_matrix(12, 10, m=3, layers=2)
print("Top layer:   ", m2[0])
print("Bottom layer:", m2[1])

print()
print("Interpretation: +1=A forward, -1=A return, +2=B forward, -2=B return, etc.")

---

## `winding_factor_sos`: Winding Factor via Phasor Sum

### Theory

The winding factor for mechanical harmonic ν is the magnitude of the normalised phasor sum of all conductors of one phase:

$$
k_{w\nu} = \frac{1}{N_k} \left| \sum_{i,\text{lyr}} s_{i,\text{lyr}} \cdot e^{j\nu \frac{2\pi i}{Q}} \right|
$$

where $s_{i,\text{lyr}} \in \{+1, -1\}$ is the conductor direction, and $N_k$ is the total conductor count per phase. The result is averaged over all m phases.

This phasor method works for **all winding types** — integer-slot, fractional-slot, and FSCW — without needing separate kp and kd formulas.

**References:** Müller, Vogt & Ponick (2008), eq. (3.65); Bianchi & Bolognani (2002).

In [ ]:
#| export
def winding_factor_sos(
    nu: int,
    Q: int,
    P: int,
    m: int = 3,
    layers: int = 1,
    w: int | None = None,
) -> float:
    """
    Winding factor |kw(ν)| via the star-of-slots phasor method.

    Works for integer-slot, fractional-slot, and FSCW windings.

    Parameters
    ----------
    nu     : int         Mechanical harmonic order
    Q      : int         Number of stator slots
    P      : int         Number of poles
    m      : int         Number of phases (default 3)
    layers : int         1 or 2 (default 1)
    w      : int | None  Coil span. None → Q // (P // 2).

    Returns
    -------
    float   |kw(ν)| ∈ [0, 1]

    Examples
    --------
    >>> round(winding_factor_sos(5, 12, 10, layers=2), 4)
    0.933
    >>> round(winding_factor_sos(4, 12, 8, layers=2), 4)
    0.866
    """
    matrix = build_coil_matrix(Q, P, m, layers, w)
    slots = np.arange(Q, dtype=np.float64)
    phasors = np.exp(1j * nu * 2.0 * np.pi * slots / Q)

    kw_vals = []
    for k in range(m):
        ph = k + 1
        total = 0j
        n_cond = 0
        for lyr in range(layers):
            fwd = matrix[lyr] == +ph
            ret = matrix[lyr] == -ph
            total += phasors[fwd].sum() - phasors[ret].sum()
            n_cond += int(fwd.sum()) + int(ret.sum())
        if n_cond > 0:
            kw_vals.append(abs(total) / n_cond)

    return float(np.mean(kw_vals)) if kw_vals else 0.0

In [ ]:
from emachines.winding.sos import winding_factor_sos

configs = [
    (12, 10, 2, "12s/10p FSCW  — ν=5 working"),
    (12,  8, 2, "12s/8p  FSCW  — ν=4 working"),
    ( 9,  8, 2, "9s/8p   FSCW  — ν=4 working"),
    (24,  4, 2, "24s/4p  ISW   — ν=2 working"),
]

print(f"{'Configuration':<35}  kw at working ν")
print("-" * 55)
for Q, P, layers, label in configs:
    p = P // 2
    kw = winding_factor_sos(p, Q, P, layers=layers)
    print(f"  {label:<33}  {kw:.4f}")

---

## `check_symmetry`: Validate Winding Balance

### Theory

A valid three-phase winding must be **balanced**: each phase must carry the same number of conductors, and the total slot count must divide evenly.

The check verifies:
1. Q is divisible by m (equal slots per phase).
2. The total period t = gcd(Q, p) > 0.
3. The number of conductors per phase is equal across all m phases.

**Reference:** Pyrhönen et al. (2008), §2.1.

In [ ]:
#| export
def check_symmetry(Q: int, P: int, m: int = 3, layers: int = 1) -> dict:
    """
    Check if the winding has the expected symmetry.

    Parameters
    ----------
    Q      : int   Number of stator slots
    P      : int   Number of poles
    m      : int   Number of phases (default 3)
    layers : int   1 or 2 (default 1)

    Returns
    -------
    dict with keys:
        balanced     : bool   True if all phases carry equal conductors
        conductors   : list   Conductor count per phase [n_A, n_B, n_C, …]
        periodicity  : int    Winding periodicity t = gcd(Q, p)
        q            : Fraction   Slots per pole per phase
        valid        : bool   True if balanced AND Q % m == 0

    Examples
    --------
    >>> r = check_symmetry(12, 10, layers=2)
    >>> r['balanced'], r['valid']
    (True, True)
    >>> r = check_symmetry(12, 4, layers=1)
    >>> r['balanced']
    True
    """
    p = P // 2
    t = gcd(Q, p)
    q = Fraction(Q, 2 * m * p)

    matrix = build_coil_matrix(Q, P, m, layers)
    conductors = []
    for k in range(m):
        ph = k + 1
        cnt = sum(
            int((matrix[lyr] == +ph).sum()) + int((matrix[lyr] == -ph).sum())
            for lyr in range(layers)
        )
        conductors.append(cnt)

    balanced = len(set(conductors)) == 1
    valid = balanced and (Q % m == 0)

    return dict(balanced=balanced, conductors=conductors, periodicity=t, q=q, valid=valid)

In [ ]:
from emachines.winding.sos import check_symmetry

for Q, P in [(12, 10), (12, 8), (9, 8), (24, 4), (6, 2)]:
    r = check_symmetry(Q, P, layers=2)
    status = "✓" if r['valid'] else "✗"
    print(f"  {status} {Q}s/{P}p  balanced={r['balanced']}  conductors={r['conductors']}  q={r['q']}  t={r['periodicity']}")

---

## `get_valid_coil_spans`: Valid Coil Span Enumeration

### Theory

Not all coil spans produce a valid balanced winding. The valid spans are those that maintain winding symmetry — specifically those where the span does not coincide with a phase boundary, causing phase short-circuiting.

For FSCW (q < 1), the only valid span is typically w = 1 (tooth coil).
For integer-slot windings, spans from 1 to `full_pitch` (= Q//p) are candidates; the full-pitch span maximises kw.

The function returns all spans that produce a balanced winding (equal conductors per phase).

In [ ]:
#| export
def get_valid_coil_spans(Q: int, P: int, m: int = 3) -> list[int]:
    """
    Return all coil spans (in slots) that yield a balanced winding.

    Evaluates spans 1 … Q//(P//2) and keeps those where all m phases
    carry an equal number of conductors in the double-layer assignment.

    Parameters
    ----------
    Q : int   Number of stator slots
    P : int   Number of poles
    m : int   Number of phases (default 3)

    Returns
    -------
    list[int]   Valid coil spans in ascending order (always non-empty).

    Examples
    --------
    >>> get_valid_coil_spans(12, 10)
    [1, 2]
    >>> get_valid_coil_spans(24, 4)
    [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
    """
    valid = []
    max_span = Q // P or 1  # pole pitch floor; for FSCW this is 1
    # Also check spans up to electrical period for ISW
    p = P // 2
    elec_period = Q // p
    for w in range(1, max(max_span, elec_period) + 1):
        matrix = build_coil_matrix(Q, P, m, layers=2, w=w)
        counts = []
        for k in range(m):
            ph = k + 1
            cnt = (int((matrix[0] == +ph).sum()) + int((matrix[0] == -ph).sum()) +
                   int((matrix[1] == +ph).sum()) + int((matrix[1] == -ph).sum()))
            counts.append(cnt)
        if len(set(counts)) == 1:
            valid.append(w)

    return valid if valid else [1]

In [ ]:
from emachines.winding.sos import get_valid_coil_spans

for Q, P in [(12, 10), (12, 8), (9, 8), (24, 4), (36, 6)]:
    spans = get_valid_coil_spans(Q, P)
    print(f"  {Q}s/{P}p  valid spans: {spans}  (full-pitch = {Q // (P//2)})")

---

## `is_valid_combination`: Feasibility Check

Returns `True` if a (Q, P, m) combination can produce a balanced m-phase winding.
This is the entry-point for design sweeps and UI filtering.

In [ ]:
#| export
def is_valid_combination(Q: int, P: int, m: int = 3) -> bool:
    """
    Return True if (Q, P, m) supports a balanced m-phase winding.

    A combination is valid when:
    - P is even and ≥ 2
    - Q and m have a common factor ≥ 1 (at least one slot per phase)
    - A double-layer coil matrix with the default span is balanced

    Parameters
    ----------
    Q : int   Number of stator slots
    P : int   Number of poles
    m : int   Number of phases (default 3)

    Returns
    -------
    bool

    Examples
    --------
    >>> is_valid_combination(12, 10)
    True
    >>> is_valid_combination(12, 10, m=5)
    False
    >>> is_valid_combination(7, 4)
    False
    """
    if P < 2 or P % 2 != 0:
        return False
    if Q < m:
        return False
    try:
        result = check_symmetry(Q, P, m, layers=1)
        return bool(result['valid'])
    except Exception:
        return False

In [ ]:
from emachines.winding.sos import is_valid_combination

print("Feasibility sweep (Q=6…18, P=2…12, m=3):")
for Q in range(6, 19, 3):
    row = []
    for P in range(2, 13, 2):
        v = is_valid_combination(Q, P)
        row.append(f"P={P}:{'✓' if v else '✗'}")
    print(f"  Q={Q:2d}: {' '.join(row)}")

---

## References

1. Pyrhönen, J., Jokinen, T., & Hrabovcová, V. (2008). *Design of Rotating Electrical Machines*. Wiley. §2.1–2.4.
2. Hanselman, D.C. (2003). *Brushless Permanent Magnet Motor Design*, 2nd ed. Writer's Collective. Ch. 4.
3. Bianchi, N., & Bolognani, S. (2002). Design techniques for reducing cogging torque in surface-mounted PM motors. *IEEE Trans. Ind. Appl.*, 38(5), 1259–1265.
4. Müller, G., Vogt, K., & Ponick, B. (2008). *Berechnung elektrischer Maschinen*. Wiley-VCH. §3.4.
5. SWAT-EM: [https://github.com/bayonet222/swat-em](https://github.com/bayonet222/swat-em)

---

## Tests

In [ ]:
#| hide
import math
import numpy as np
from emachines.winding.sos import (
    get_basic_params, build_star_of_slots, assign_phases,
    build_coil_matrix, winding_factor_sos, check_symmetry,
    get_valid_coil_spans, is_valid_combination,
)

# ── get_basic_params ──────────────────────────────────────────────────────────
p = get_basic_params(12, 10)
assert p['p'] == 5 and p['t'] == 1, f"12s/10p params wrong: {p}"
p24 = get_basic_params(24, 4)
assert p24['p'] == 2 and p24['q'] == 2, f"24s/4p params wrong: {p24}"
print("✓ get_basic_params")

# ── build_star_of_slots ───────────────────────────────────────────────────────
angles = build_star_of_slots(6, 2)
assert angles.shape == (6,), "shape"
assert math.isclose(angles[0], 0.0) and math.isclose(angles[3], math.pi), "6s/2p angles"
print("✓ build_star_of_slots")

# ── assign_phases ─────────────────────────────────────────────────────────────
codes = assign_phases(6, 2, m=3)
assert list(codes) == [1, 2, 3, -1, -2, -3], f"6s/2p assignment wrong: {codes}"
codes12 = assign_phases(12, 10)
assert len(codes12) == 12, "length"
assert set(abs(c) for c in codes12) == {1, 2, 3}, "phases 1-3 only"
print("✓ assign_phases")

# ── build_coil_matrix ─────────────────────────────────────────────────────────
m1 = build_coil_matrix(6, 2, layers=1)
assert m1.shape == (1, 6), f"shape {m1.shape}"
assert m1.tolist() == [[1, 2, 3, -1, -2, -3]], f"6s/2p SL wrong: {m1}"
m2 = build_coil_matrix(12, 10, layers=2)
assert m2.shape == (2, 12), f"shape {m2.shape}"
# Each layer should have all 12 slots occupied (no zeros for DL FSCW)
assert not (m2 == 0).any(), "DL matrix has empty slots"
print("✓ build_coil_matrix")

# ── winding_factor_sos ────────────────────────────────────────────────────────
kw_12_10 = winding_factor_sos(5, 12, 10, layers=2)
assert math.isclose(kw_12_10, 0.9330, abs_tol=1e-3), f"12s/10p kw(5)={kw_12_10:.4f}"
kw_12_8  = winding_factor_sos(4, 12, 8, layers=2)
assert math.isclose(kw_12_8, 0.8660, abs_tol=1e-3), f"12s/8p kw(4)={kw_12_8:.4f}"
assert 0 < winding_factor_sos(2, 24, 4, layers=1) <= 1.0, "24s/4p kw out of range"
print("✓ winding_factor_sos")

# ── check_symmetry ────────────────────────────────────────────────────────────
r = check_symmetry(12, 10, layers=2)
assert r['balanced'] and r['valid'], f"12s/10p should be valid: {r}"
r_invalid = check_symmetry(7, 4)
assert not r_invalid['valid'], "7s/4p should be invalid"
print("✓ check_symmetry")

# ── get_valid_coil_spans ──────────────────────────────────────────────────────
spans_12_10 = get_valid_coil_spans(12, 10)
assert 1 in spans_12_10, f"12s/10p valid spans should include 1: {spans_12_10}"
spans_24_4 = get_valid_coil_spans(24, 4)
assert 12 in spans_24_4, f"full-pitch (12) should be valid for 24s/4p"
print("✓ get_valid_coil_spans")

# ── is_valid_combination ──────────────────────────────────────────────────────
assert is_valid_combination(12, 10) == True
assert is_valid_combination(7, 4) == False
assert is_valid_combination(12, 10, m=5) == False
print("✓ is_valid_combination")

print()
print("✓ All SOS tests passed")

# ── Parameterised checks across reference combinations ────────────────────────
_ref_combos = [
    # (Q,  P, layers, nu_working, kw_expected, abs_tol, label)
    (12, 10, 2, 5, 0.9330, 1e-3, '12s/10p FSCW'),
    (12, 14, 2, 7, 0.9330, 1e-3, '12s/14p FSCW'),
    (12,  8, 2, 4, 0.8660, 1e-3, '12s/8p  FSCW'),
    (24,  4, 2, 2, 0.9659, 1e-3, '24s/4p  ISW'),
    (36,  8, 2, 4, 0.9452, 1e-3, '36s/8p  ISW'),
]

for Q, P, layers, nu_w, kw_exp, tol, label in _ref_combos:
    # is_valid
    assert is_valid_combination(Q, P), f"{label}: expected valid combination"
    # check_symmetry balanced
    sym = check_symmetry(Q, P, layers=layers)
    assert sym["balanced"], f"{label}: expected balanced winding"
    # build_coil_matrix shape
    mat = build_coil_matrix(Q, P, layers=layers)
    assert mat.shape == (layers, Q), f"{label}: matrix shape {mat.shape}"
    # winding_factor_sos at working harmonic
    kw = winding_factor_sos(nu_w, Q, P, layers=layers)
    assert math.isclose(kw, kw_exp, abs_tol=tol),         f"{label}: kw(nu={nu_w})={kw:.4f}, expected {kw_exp}"
    print(f"pass  {label}: kw(nu={nu_w})={kw:.4f}")
